<a href="https://colab.research.google.com/github/tffnnnnnn/Machine_Learning_MASnI3/blob/main/%E6%90%9C%E5%B0%8BNOMAD_MASnI3%E7%9A%84DFT%E8%B3%87%E6%96%99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

確認API是否正常運作，且可讀取622筆公開資料

In [1]:

import requests

# NOMAD 官方 API
API_URL = "https://nomad-lab.eu/prod/v1/api/v1"

# MASnI3 化學式：CH6I3NSn
query = {
    "results.material.chemical_formula_hill": "CH6I3NSn",
    "results.method.method_name": "DFT"
}

# 先確認搜尋數量與回傳格式
payload = {
    "owner": "public",
    "query": query,
    "pagination": {
        "page_size": 5
    }
}

response = requests.post(
    f"{API_URL}/entries/query",
    json=payload,
    timeout=60
)

response.raise_for_status()
result = response.json()

print("=== NOMAD MASnI3 DFT Search ===")
print("Total matching entries:", result["pagination"]["total"])
print("Returned entries:", len(result["data"]))

print("\n=== First 5 Entries ===")

for i, entry in enumerate(result["data"], 1):
    print(f"\nEntry {i}")
    print("Entry ID:", entry.get("entry_id"))

    material = entry.get("results", {}).get("material", {})
    print("Formula:", material.get("chemical_formula_hill"))

    method = entry.get("results", {}).get("method", {})
    print("Method:", method.get("method_name"))


=== NOMAD MASnI3 DFT Search ===
Total matching entries: 622
Returned entries: 5

=== First 5 Entries ===

Entry 1
Entry ID: --aVj0_fQXVFQuzQ7FWUL8pxX_C8
Formula: CH6I3NSn
Method: DFT

Entry 2
Entry ID: -SCdLmiM-XK_5m6JvmNckuodlpLa
Formula: CH6I3NSn
Method: DFT

Entry 3
Entry ID: -areUk8CdbRLUJSgEGt6VK116A3L
Formula: CH6I3NSn
Method: DFT

Entry 4
Entry ID: -bc76c6qQnD_laqNST6-8DVJkY8y
Formula: CH6I3NSn
Method: DFT

Entry 5
Entry ID: -i5blgHtFqEoLVzuNZtqcyl4rTHr
Formula: CH6I3NSn
Method: DFT


In [2]:
import requests

all_entries = []
page_after = None

while True:
    payload = {
        "owner": "public",
        "query": query,
        "pagination": {
            "page_size": 100
        },
        "required": {
            "include": [
                "entry_id",
                "upload_id",
                "results.material.chemical_formula_hill",
                "results.method.method_name"
            ]
        }
    }

    if page_after is not None:
        payload["pagination"]["page_after_value"] = page_after

    response = requests.post(
        f"{API_URL}/entries/query",
        json=payload,
        timeout=60
    )

    response.raise_for_status()
    result = response.json()

    all_entries.extend(result["data"])

    print(f"Retrieved: {len(all_entries)} entries")

    page_after = result["pagination"].get(
        "next_page_after_value"
    )

    if not page_after:
        break

# 確認搜尋結果
entry_ids = [entry["entry_id"] for entry in all_entries]

print("\n=== NOMAD Dataset Verification ===")
print("Total retrieved:", len(all_entries))
print("Unique Entry IDs:", len(set(entry_ids)))
print("Expected total:", result["pagination"]["total"])

if len(all_entries) == result["pagination"]["total"]:
    print("\nAll MASnI3 DFT entries retrieved successfully!")
else:
    print("\nWarning: Retrieved count differs from expected total.")

Retrieved: 100 entries
Retrieved: 200 entries
Retrieved: 300 entries
Retrieved: 400 entries
Retrieved: 500 entries
Retrieved: 600 entries
Retrieved: 622 entries

=== NOMAD Dataset Verification ===
Total retrieved: 622
Unique Entry IDs: 622
Expected total: 622

All MASnI3 DFT entries retrieved successfully!


檢查 DFT Archive 資料格式

In [3]:
import requests

# 取得第一筆 MASnI3 DFT 計算的 Archive
test_entry_id = entry_ids[0]

url = f"{API_URL}/entries/{test_entry_id}/archive"

response = requests.get(url, timeout=60)
response.raise_for_status()

archive_response = response.json()
archive = archive_response.get("data", archive_response)

print("=== NOMAD DFT Archive Inspection ===")
print("Entry ID:", test_entry_id)

print("\n=== Top-level Keys ===")
print(list(archive.keys()))

print("\n=== Run Information ===")

runs = archive.get("run", [])

print("Number of runs:", len(runs))

if runs:
    run = runs[0]

    print("Run keys:", list(run.keys()))
    print("Number of systems:", len(run.get("system", [])))
    print("Number of calculations:", len(run.get("calculation", [])))

    calculations = run.get("calculation", [])

    if calculations:
        calc = calculations[-1]

        print("\n=== Calculation Keys ===")
        print(list(calc.keys()))

        print("\n=== DFT Data Availability ===")
        print("Energy:", "energy" in calc)
        print("Forces:", "forces" in calc)
        print("Stress:", "stress" in calc)

else:
    print("No run section found.")
    print("We need to inspect the archive structure.")

=== NOMAD DFT Archive Inspection ===
Entry ID: --aVj0_fQXVFQuzQ7FWUL8pxX_C8

=== Top-level Keys ===
['entry_id', 'upload_id', 'parser_name', 'archive']

=== Run Information ===
Number of runs: 0
No run section found.
We need to inspect the archive structure.


讀取一筆資料

In [4]:
# 修正 NOMAD Archive 資料層級
archive = archive_response["data"]["archive"]

print("=== Corrected NOMAD Archive Inspection ===")

print("\nArchive keys:")
print(list(archive.keys()))

runs = archive.get("run", [])

print("\nNumber of runs:", len(runs))

if runs:
    run = runs[0]

    print("\nRun keys:")
    print(list(run.keys()))

    print("\nNumber of systems:", len(run.get("system", [])))
    print("Number of calculations:", len(run.get("calculation", [])))

    calculations = run.get("calculation", [])

    if calculations:
        calc = calculations[-1]

        print("\n=== DFT Data Availability ===")

        for key in ["energy", "forces", "stress"]:
            value = calc.get(key)

            print(f"{key}: {'Available' if value is not None else 'Not found'}")

        print("\nCalculation keys:")
        print(list(calc.keys()))
else:
    print("No run section found in archive.")

=== Corrected NOMAD Archive Inspection ===

Archive keys:
['processing_logs', 'run', 'workflow', 'workflow2', 'metadata', 'results', 'm_ref_archives']

Number of runs: 1

Run keys:
['x_fhi_aims_number_of_tasks', 'x_fhi_aims_program_compilation_date', 'x_fhi_aims_program_compilation_time', 'x_fhi_aims_program_execution_date', 'x_fhi_aims_program_execution_time', 'x_fhi_aims_section_parallel_tasks', 'program', 'time_run', 'method', 'system', 'calculation']

Number of systems: 1
Number of calculations: 1

=== DFT Data Availability ===
energy: Available
forces: Not found
stress: Not found

Calculation keys:
['system_ref', 'method_ref', 'calculation_converged', 'x_fhi_aims_energy_C_LDA', 'x_fhi_aims_energy_X_LDA', 'x_fhi_aims_calculation_md', 'n_scf_iterations', 'energy', 'eigenvalues', 'scf_iteration']


讀取662筆資料

In [5]:
import requests
import pandas as pd
import time
from collections import Counter

# =====================================================
# NOMAD MASnI3 DFT Dataset Analysis
# =====================================================

records = []
failed_entries = []

session = requests.Session()

def has_value(section):
    """檢查 NOMAD 物理量是否包含非空數值。"""
    if section is None:
        return False

    if isinstance(section, dict):
        if "total" in section:
            return has_value(section["total"])
        if "value" in section:
            return has_value(section["value"])
        return False

    if isinstance(section, (list, tuple)):
        return len(section) > 0

    return True


for i, entry_id in enumerate(entry_ids, 1):

    try:
        url = f"{API_URL}/entries/{entry_id}/archive"

        response = session.get(url, timeout=60)
        response.raise_for_status()

        result = response.json()
        archive = result["data"]["archive"]

        has_structure = False
        has_energy = False
        has_forces = False
        has_stress = False

        program_names = set()
        calculation_count = 0

        for run in archive.get("run", []):

            program = run.get("program", {})
            program_names.add(program.get("name", "Unknown"))

            for system in run.get("system", []):
                atoms = system.get("atoms", {})

                if (
                    atoms.get("positions") is not None
                    and atoms.get("lattice_vectors") is not None
                    and atoms.get("labels") is not None
                ):
                    has_structure = True

            for calc in run.get("calculation", []):

                calculation_count += 1

                if has_value(calc.get("energy")):
                    has_energy = True

                if has_value(calc.get("forces")):
                    has_forces = True

                if has_value(calc.get("stress")):
                    has_stress = True

        records.append({
            "entry_id": entry_id,
            "program": ", ".join(sorted(program_names)),
            "n_calculations": calculation_count,
            "structure": has_structure,
            "energy": has_energy,
            "forces": has_forces,
            "stress": has_stress,
            "energy_forces": has_energy and has_forces,
            "complete_EFS": (
                has_structure
                and has_energy
                and has_forces
                and has_stress
            )
        })

    except Exception as e:
        failed_entries.append({
            "entry_id": entry_id,
            "error": str(e)
        })

    # 每50筆顯示進度並保存
    if i % 50 == 0 or i == len(entry_ids):
        print(f"Processed: {i}/{len(entry_ids)}")

        pd.DataFrame(records).to_csv(
            "MASnI3_DFT_availability.csv",
            index=False
        )

    # 避免短時間向 NOMAD 發送過多請求
    time.sleep(0.2)


# =====================================================
# Summary
# =====================================================

df = pd.DataFrame(records)

print("\n=== MASnI3 DFT Dataset Summary ===")

print("Total searched:", len(entry_ids))
print("Successfully analyzed:", len(df))
print("Failed entries:", len(failed_entries))

print("\n=== Data Availability ===")

for column in [
    "structure",
    "energy",
    "forces",
    "stress",
    "energy_forces",
    "complete_EFS"
]:
    print(f"{column}: {int(df[column].sum())}")

print("\n=== Calculation Programs ===")
print(df["program"].value_counts())

print("\n=== Calculation Count ===")
print("Total calculations:", int(df["n_calculations"].sum()))

print("\nResults saved:")
print("MASnI3_DFT_availability.csv")

if failed_entries:
    pd.DataFrame(failed_entries).to_csv(
        "MASnI3_failed_entries.csv",
        index=False
    )
    print("Failed entries saved: MASnI3_failed_entries.csv")

Processed: 50/622
Processed: 100/622
Processed: 150/622
Processed: 200/622
Processed: 250/622
Processed: 300/622
Processed: 350/622
Processed: 400/622
Processed: 450/622
Processed: 500/622
Processed: 550/622
Processed: 600/622
Processed: 622/622

=== MASnI3 DFT Dataset Summary ===
Total searched: 622
Successfully analyzed: 421
Failed entries: 201

=== Data Availability ===
structure: 421
energy: 421
forces: 29
stress: 31
energy_forces: 29
complete_EFS: 29

=== Calculation Programs ===
program
FHI-aims            390
VASP                 29
Quantum Espresso      2
Name: count, dtype: int64

=== Calculation Count ===
Total calculations: 484

Results saved:
MASnI3_DFT_availability.csv
Failed entries saved: MASnI3_failed_entries.csv


確認201筆無法讀取原因

原因：API請求頻率過於頻繁

In [6]:
import pandas as pd
from collections import Counter

# =====================================================
# Step 5: Analyze Failed NOMAD Entries
# =====================================================

failed_df = pd.DataFrame(failed_entries)

print("=== Failed Entries Analysis ===")
print("Total failed entries:", len(failed_df))

# 依錯誤類型分類
def classify_error(error):
    error = str(error)

    if "429" in error:
        return "Rate limit (429)"
    elif "404" in error:
        return "Not found (404)"
    elif "403" in error:
        return "Access denied (403)"
    elif "500" in error:
        return "Server error (500)"
    elif "502" in error:
        return "Bad gateway (502)"
    elif "503" in error:
        return "Service unavailable (503)"
    elif "504" in error:
        return "Gateway timeout (504)"
    elif "timed out" in error.lower():
        return "Timeout"
    elif "connection" in error.lower():
        return "Connection error"
    elif "archive" in error.lower():
        return "Archive error"
    else:
        return "Other error"

failed_df["error_type"] = failed_df["error"].apply(classify_error)

print("\n=== Error Type Distribution ===")
print(failed_df["error_type"].value_counts())

print("\n=== Example Errors ===")

for error_type, group in failed_df.groupby("error_type"):
    print(f"\n[{error_type}]")
    print("Example Entry ID:", group.iloc[0]["entry_id"])
    print("Error message:", group.iloc[0]["error"][:300])

print("\n=== Successful Entries by Program ===")
print(df["program"].value_counts())

print("\n=== Complete EFS Entries by Program ===")
print(
    df[df["complete_EFS"] == True]["program"].value_counts()
)

=== Failed Entries Analysis ===
Total failed entries: 201

=== Error Type Distribution ===
error_type
Rate limit (429)    201
Name: count, dtype: int64

=== Example Errors ===

[Rate limit (429)]
Example Entry ID: 2GygyOufhgnqndE9TbHfDo7RWQcq
Error message: 429 Client Error: Too Many Requests for url: https://nomad-lab.eu/prod/v1/api/v1/entries/2GygyOufhgnqndE9TbHfDo7RWQcq/archive

=== Successful Entries by Program ===
program
FHI-aims            390
VASP                 29
Quantum Espresso      2
Name: count, dtype: int64

=== Complete EFS Entries by Program ===
program
VASP    29
Name: count, dtype: int64
